# 08 · Distributed: all_reduce and DDP with torchrun

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/pytorch/08_ddp_torchrun.ipynb) &nbsp; 📖 Concept deck: [Distributed training in PyTorch](https://nilesh-salpe.github.io/visual-ai/pytorch/08-distributed/) &nbsp; ⏱️ Suggested time: **35 min**

**You will build:** launching two CPU processes with `torchrun` and the gloo backend, averaging gradients by hand with `dist.all_reduce`, and checking that `DistributedDataParallel` gives exactly the same result.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0)
print("PyTorch", torch.__version__)

import subprocess, sys, textwrap, inspect, os, json, tempfile
def run_torchrun(script, nproc=2):
    """Write `script` to a temp file and run it with torchrun on CPU (gloo). Returns rank 0's printed JSON."""
    d = tempfile.mkdtemp(); path = os.path.join(d, "job.py"); open(path, "w").write(textwrap.dedent(script))
    r = subprocess.run([sys.executable, "-m", "torch.distributed.run", "--nproc_per_node", str(nproc), "--master_port", "29511", path], capture_output=True, text=True, timeout=300)
    lines = [l for l in r.stdout.splitlines() if l.startswith("{")]
    if not lines: raise RuntimeError(r.stderr[-2000:])
    return json.loads(lines[0])
COMMON = """
import json, torch, torch.nn as nn, torch.nn.functional as F, torch.distributed as dist
dist.init_process_group("gloo")
rank, world = dist.get_rank(), dist.get_world_size()
torch.manual_seed(0); model = nn.Linear(4, 1)
g = torch.Generator().manual_seed(1); X, Y = torch.randn(16, 4, generator=g), torch.randn(16, 1, generator=g)
xs, ys = X.chunk(world)[rank], Y.chunk(world)[rank]
"""

## Task 1: Average gradients with all_reduce

Write the body of `average_gradients(model, world)` as source code (a string) that, for every parameter, sums `p.grad` across ranks with `dist.all_reduce(p.grad, op=dist.ReduceOp.SUM)` and divides by `world`.
The test runs it under `torchrun` with 2 processes, each holding half the batch, and checks the result equals the full-batch gradient.

In [ ]:
def average_gradients_src():
    return """
def average_gradients(model, world):
    # TODO
    raise NotImplementedError
"""

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_average_gradients_src(f):
    src = f()
    if "NotImplementedError" in src: raise NotImplementedError
    out = run_torchrun(COMMON + textwrap.dedent(src) + """
F.mse_loss(model(xs), ys).backward()
average_gradients(model, world)
ref = nn.Linear(4, 1); torch.manual_seed(0); ref = nn.Linear(4, 1); F.mse_loss(ref(X), Y).backward()
ok = all(torch.allclose(p.grad, q.grad, atol=1e-6) for p, q in zip(model.parameters(), ref.parameters()))
if rank == 0: print(json.dumps({"ok": ok}))
dist.destroy_process_group()
""")
    assert out["ok"], "averaged gradients must equal the full-batch gradient"

check("average_gradients_src", _test_average_gradients_src, average_gradients_src)

## Task 2: DistributedDataParallel does it for you

Now wrap the model: `ddp = torch.nn.parallel.DistributedDataParallel(model)`, run forward/backward on the local shard, and report whether every parameter's gradient equals the full-batch gradient. Return the script body as a string that prints `{"ok": …}` on rank 0.

In [ ]:
def ddp_matches_src():
    return """
raise NotImplementedError
"""

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_ddp_matches_src(f):
    src = f()
    if "NotImplementedError" in src: raise NotImplementedError
    out = run_torchrun(COMMON + textwrap.dedent(src))
    assert out["ok"], "DDP gradients must equal the full-batch gradient"

check("ddp_matches_src", _test_ddp_matches_src, ddp_matches_src)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Average gradients with all_reduce
def average_gradients_src_solution():
    return """
def average_gradients(model, world):
    for p in model.parameters():
        dist.all_reduce(p.grad, op=dist.ReduceOp.SUM)
        p.grad /= world
"""

check("average_gradients_src (reference)", _test_average_gradients_src, average_gradients_src_solution)

In [ ]:
# @title Solution for Task 2: DistributedDataParallel does it for you
def ddp_matches_src_solution():
    return """
ddp = torch.nn.parallel.DistributedDataParallel(model)
F.mse_loss(ddp(xs), ys).backward()
torch.manual_seed(0); ref = nn.Linear(4, 1); F.mse_loss(ref(X), Y).backward()
ok = all(torch.allclose(p.grad, q.grad, atol=1e-6) for p, q in zip(model.parameters(), ref.parameters()))
if rank == 0: print(json.dumps({"ok": ok}))
dist.destroy_process_group()
"""

check("ddp_matches_src (reference)", _test_ddp_matches_src, ddp_matches_src_solution)

---
## Experiments
**What torchrun sets up.** Each process gets environment variables (`RANK`, `LOCAL_RANK`, `WORLD_SIZE`, `MASTER_ADDR`, `MASTER_PORT`) that `init_process_group` reads. On GPUs you would use the "nccl" backend and set the device from `LOCAL_RANK`; FSDP2 replaces DDP with `fully_shard(layer)` on each block and then on the root module.

In [ ]:
out = run_torchrun("""
import os, json, torch.distributed as dist
dist.init_process_group("gloo")
env = {k: os.environ[k] for k in ["RANK", "LOCAL_RANK", "WORLD_SIZE", "MASTER_ADDR", "MASTER_PORT"]}
if dist.get_rank() == 0: print(json.dumps(env))
dist.destroy_process_group()
""")
print("rank 0 environment:", out)

---
## Takeaways
* `torchrun` launches one process per device and sets `RANK`, `LOCAL_RANK` and `WORLD_SIZE`; `init_process_group("nccl" | "gloo")` connects them.
* Data parallelism = each rank computes gradients on its shard, then all-reduce averages them; DDP does this in buckets (25 MB by default) overlapped with backward.
* FSDP2 (`fully_shard`) applies the same idea with sharded parameters: all-gather before use, reduce-scatter gradients after.